# Decision Tranformer Training

An offline reinforcement learning, where the agent only uses data collected from other agents or human demostration. The Decision Tansformer model uses the transformer algorithm, a sequence modeling agorithm that given a desired return, past states amd action, generate future actions to achive this desired return.Generative trajectory modeling replaces training a policy using conventional RL methods.

## Dependencies for Mujogo environment

In [ ]:
#dependencies with EGL-enabled mujoco-py
!apt-get install -y libgl1-mesa-glx libosmesa6-dev patchelf
!pip install gymnasium[mujoco] mujoco==3.1.3 imageio imageio[ffmpeg] pyvirtualdisplay

In [ ]:
!pip install transformers

In [ ]:
!pip install gymnasium[mujoco]   # Multi-Joint Dynamics with Contrast framework

In [ ]:
!pip install xvfbwrapper
!pip install imageio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/ColabNotebooks/DeepRL/offline_rl/

Mounted at /content/drive
/content/drive/MyDrive/ColabNotebooks/DeepRL/offline_rl


## Package import

In [ ]:
import torch
import numpy as np

import os
# use EGL backend by setting an environment variable before makeing the env
os.environ["MUJOCO_GL"] = "egl"

import gymnasium as gym
from gymnasium.wrappers import RecordVideo

env = gym.make("Hopper-v5", render_mode="rgb_array")
# to ensure every episode is recorded use episode_trigger=lambda x: True
env = RecordVideo(env, "./videos", episode_trigger=lambda x: True)


/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/ColabNotebooks/DeepRL/offline_rl/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [ ]:
# start a virtual display, headless OpenGL, setting up a dummy display so MuJoGo can render offscreen
from pyvirtualdisplay import Display

Display(visible=0, size=(1400, 900)).start()

## Loading pre-trained Decision Transformer Model

In [ ]:
from transformers import DecisionTransformerModel

model_name = "edbeeching/decision-transformer-gym-hopper-expert"
#model_name = "edbeeching/decision-transformer-gym-halfcheetah-medium-replay"
model = DecisionTransformerModel.from_pretrained(model_name)
device = "cpu"
model = model.to(device)

print(list(model.encoder.wpe.parameters()))

config.json:   0%|          | 0.00/945 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/6.60M [00:00<?, ?B/s]

[Parameter containing:
tensor([[-0., -0., 0.,  ..., 0., 0., 0.],
        [0., -0., -0.,  ..., 0., 0., -0.],
        [-0., 0., -0.,  ..., 0., 0., -0.],
        ...,
        [0., -0., -0.,  ..., 0., 0., 0.],
        [0., -0., 0.,  ..., 0., 0., 0.],
        [0., -0., 0.,  ..., 0., 0., -0.]], requires_grad=True)]


## Load Environment

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo

#for i in gym.envs.registry.keys():
  #print(i)

env = gym.make("Hopper-v5", render_mode='rgb_array')
#env = gym.make("HalfCheetah-v5", render_mode="rgb_array")
state_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]

print("state_dim:", state_dim, "act_dim:", act_dim)


state_dim: 11 act_dim: 3


## State normalizatiom

As scaling returns, normalizing states is required before using the model for inference in the environment, it is needed to normalize the current state using the pre-computed state_mean and state_std values before feeding it into the model. This ensures that the input to the model is consistent with the data it was trained on, which is essential for it to make accurate predictions.

In [ ]:
scale = 1000.0                 # since reward values can vary across different environments, they need to be scalled
TARGET_RETURN = 3600 / scale   # evaluation is conditioned on a specified return of 3600, scaled accordingly, which
                               # means the model is trying to take action that lead to a scaled return of 3.6

# numpy arrays containing mean and std computed for each of the 11th dimensions of the state space of the training dataset, model card for each model.
state_mean = np.array(
    [1.3490015,  -0.11208222, -0.5506444,  -0.13188992, -0.00378754,  2.6071432,
     0.02322114, -0.01626922, -0.06840388, -0.05183131,  0.04272673,]
)
state_std = np.array(
    [0.15980862, 0.0446214,  0.14307782, 0.17629202, 0.5912333,  0.5899924,
 1.5405099,  0.8152689,  2.0173461,  2.4107876,  5.8440027,]
)

# convert the numpy arrays into pytorch tensors
state_mean = torch.from_numpy(state_mean).to(device=device)
state_std = torch.from_numpy(state_std).to(device=device)

In [ ]:
print("Loaded mean/std shapes:", state_mean.shape, state_std.shape)

# dimension consistency
assert state_dim == model.config.state_dim, f"Mismatch: env={state_dim}, model={model.config.state_dim}"
assert act_dim == model.config.act_dim, f"Mismatch: env={act_dim}, model={model.config.act_dim}"

Loaded mean/std shapes: torch.Size([11]) torch.Size([11])


## Autoregressive prediction function

The model's prediction is conditioned on sequences of states, actions, time-steps and returns. The action for the current time step is determined by a autoregressive prediction function with a window of the previous 20 timesteps.

In [ ]:
def get_action(model, states, actions, rewards, returns_to_go, timesteps):
    """ The function uses the Decision Transformer model to predict the most likely
        action to take at the current step. It prepares and feeds the sequence of
        past states, past action and past rewards into the Decision Transformer model,
        which expects a sequence of a fixed length model.config.max_length,
        so the input sequences are either truncated or padded with 0's.
        An attention_mask is created to tell the model the actual data from
        the pedding. The function returns the predicted action for the last
        timestep in the sequence.

        Note: Even though rewards sequence is passed to the model, it is NOT used
              for conditioning the model predictions. The Decision Transformer
              conditions on the desired outcome/ return-to-go rather than how
              that outcome was accumulated though individual past rewards.  """

    states = states.reshape(1, -1, model.config.state_dim)
    actions = actions.reshape(1, -1, model.config.act_dim)
    returns_to_go = returns_to_go.reshape(1, -1, 1)
    timesteps = timesteps.reshape(1, -1)

    states = states[:, -model.config.max_length :]
    actions = actions[:, -model.config.max_length :]
    returns_to_go = returns_to_go[:, -model.config.max_length :]
    timesteps = timesteps[:, -model.config.max_length :]
    padding = model.config.max_length - states.shape[1]

    # pad all tokens to sequence length
    attention_mask = torch.cat([torch.zeros(padding), torch.ones(states.shape[1])])
    attention_mask = attention_mask.to(dtype=torch.long).reshape(1, -1)
    states = torch.cat([torch.zeros((1, padding, model.config.state_dim)), states], dim=1).float()
    actions = torch.cat([torch.zeros((1, padding, model.config.act_dim)), actions], dim=1).float()
    returns_to_go = torch.cat([torch.zeros((1, padding, 1)), returns_to_go], dim=1).float()
    timesteps = torch.cat([torch.zeros((1, padding), dtype=torch.long), timesteps], dim=1)

    # perrform the prediction
    state_preds, action_preds, return_preds = model(
        states=states,
        actions=actions,
        rewards=rewards,
        returns_to_go=returns_to_go,
        timesteps=timesteps,
        attention_mask=attention_mask,
        return_dict=False,
    )
    # predicted action for the last timestep in the sequence
    return action_preds[0, -1]

## The Rollout Pipeline

In [ ]:
# wrap environment with RecordVideo, as the agent interacts with the environment,
# a video of the episode will be recorded and saved in dir
env = RecordVideo(env, "./videos")
max_episode_length = 1000

# initialize to track the total reward accumulated during the episode
episode_return, episode_length = 0, 0

# reset the environment to initial state at the beginning of the episode
state, _ = env.reset()

# set the desired return for the entire episode; the Decision Transformer is conditioned on this cumulative reward
target_return = torch.tensor(TARGET_RETURN, device=device, dtype=torch.float32).reshape(1, 1)
# initialize aa tensor to store the sequence of states, actions encountered and rewards recived during the episode
states = torch.tensor(state, device=device).reshape(1, -1)
actions = torch.zeros((0, act_dim), device=device, dtype=torch.float32)
rewards = torch.zeros(0, device=device, dtype=torch.float32)

timesteps = torch.tensor(0, device=device, dtype=torch.long).reshape(1, 1)

# the main loop that simulated an episode by repeatedly getting an action from the Decision Transformer model
# based on the history and return-to-go = sum of the all future rewards from that timestep onwards, discounted by
# a discount factor or undiscounted sum of future rewards;
# instead of trying to predict the best action to maximize future rewards from scratch,
# Decision Transformer its told what future reward/ return-to-go is should aim for and based on observed
# history-past states, past action at past timesteps, it predicts the actions that are likely to achive that target
for t in range(max_episode_length):
    actions = torch.cat([actions, torch.zeros((1, act_dim), device=device)], dim=0)
    rewards = torch.cat([rewards, torch.zeros(1, device=device)])

    action = get_action(
        model,
        (states - state_mean) / state_std,
        actions,
        rewards,
        target_return,
        timesteps,
    )
    actions[-1] = action
    action = action.detach().cpu().numpy()

    state, reward, terminated, truncated, info = env.step(action)

    cur_state = torch.from_numpy(state).to(device=device).reshape(1, state_dim)
    states = torch.cat([states, cur_state], dim=0)
    rewards[-1] = reward

    pred_return = target_return[0, -1] - (reward / scale)
    target_return = torch.cat([target_return, pred_return.reshape(1, 1)], dim=1)
    timesteps = torch.cat([timesteps, torch.ones((1, 1), device=device, dtype=torch.long) * (t + 1)], dim=1)

    episode_return += reward
    episode_length += 1

    if terminated or truncated:
        break
print(f"episode length:, {episode_length}")
print(f"episode reward:, {episode_return:.2f}")

env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/ColabNotebooks/DeepRL/offline_rl/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


episode length:, 610
episode reward:, 2253.55


## Display Video inside Colab

In [ ]:
import io
import base64
from IPython.display import HTML

def show_video(video_path):
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + base64.b64encode(mp4).decode()
    return HTML(f'<video width=480 controls><source src="{data_url}" type="video/mp4"></video>')

# show the first recorded video
import glob
videos = glob.glob('./videos/*.mp4')
show_video(videos[0])
